## Data Cleaning and Validation

This notebook examines the quality and consistency of the
NHS England A&E dataset before statistical analysis.

The process includes:
- Checking missing values
- Identifying duplicate records
- Reviewing data types
- Standardising column names
- Validating numerical values
- Preparing a cleaned dataset

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\HP\nhs_a&e_performance analysis\Data\Raw\April_2025.csv")

# Keep the original data unchanged
clean_df = df.copy()

print("Original dataset:", clean_df.shape)

Original dataset: (199, 22)


In [10]:
import re

clean_df.columns = [re.sub(r'[^a-z0-9]+', '_', col.lower()).strip('_')
for col in clean_df.columns
]
clean_df.columns.tolist()

['period',
 'org_code',
 'parent_org',
 'org_name',
 'a_e_attendances_type_1',
 'a_e_attendances_type_2',
 'a_e_attendances_other_a_e_department',
 'a_e_attendances_booked_appointments_type_1',
 'a_e_attendances_booked_appointments_type_2',
 'a_e_attendances_booked_appointments_other_department',
 'attendances_over_4hrs_type_1',
 'attendances_over_4hrs_type_2',
 'attendances_over_4hrs_other_department',
 'attendances_over_4hrs_booked_appointments_type_1',
 'attendances_over_4hrs_booked_appointments_type_2',
 'attendances_over_4hrs_booked_appointments_other_department',
 'patients_who_have_waited_4_12_hs_from_dta_to_admission',
 'patients_who_have_waited_12_hrs_from_dta_to_admission',
 'emergency_admissions_via_a_e_type_1',
 'emergency_admissions_via_a_e_type_2',
 'emergency_admissions_via_a_e_other_a_e_department',
 'other_emergency_admissions']

In [11]:
# Identify numerical columns
numeric_cols = clean_df.columns[4:]

# Convert them to numeric
for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col],errors="coerce")

clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 22 columns):
 #   Column                                                      Non-Null Count  Dtype
---  ------                                                      --------------  -----
 0   period                                                      199 non-null    str  
 1   org_code                                                    199 non-null    str  
 2   parent_org                                                  199 non-null    str  
 3   org_name                                                    199 non-null    str  
 4   a_e_attendances_type_1                                      199 non-null    int64
 5   a_e_attendances_type_2                                      199 non-null    int64
 6   a_e_attendances_other_a_e_department                        199 non-null    int64
 7   a_e_attendances_booked_appointments_type_1                  199 non-null    int64
 8   a_e_attendances_booked_appointm

In [12]:
text_cols = ["org_code", "parent_org", "org_name"]

for col in text_cols:
    clean_df[col] = clean_df[col].astype("string").str.strip()

clean_df[text_cols].head()

,org_code,parent_org,org_name
0,NDJ,NHS ENGLAND SOUTH EAST,FIRST COMMUNITY HEALTH AND CARE CIC
1,NTV0B,NHS ENGLAND SOUTH EAST,ASHFORD WALK-IN-CENTRE
2,NTV0W,NHS ENGLAND SOUTH EAST,WOKING WALK IN CENTRE
3,Y02615,NHS ENGLAND MIDLANDS,SOUTH BIRMINGHAM GP WALK IN CENTRE
4,AQN04,NHS ENGLAND SOUTH EAST,PHL LYMINGTON UTC


In [13]:
# Check repeated provider-period combinations
duplicates = clean_df[clean_df.duplicated(subset=["period", "org_code"], keep=False)]

duplicates

,period,org_code,parent_org,org_name,a_e_attendances_type_1,a_e_attendances_type_2,a_e_attendances_other_a_e_department,a_e_attendances_booked_appointments_type_1,a_e_attendances_booked_appointments_type_2,a_e_attendances_booked_appointments_other_department,...,attendances_over_4hrs_other_department,attendances_over_4hrs_booked_appointments_type_1,attendances_over_4hrs_booked_appointments_type_2,attendances_over_4hrs_booked_appointments_other_department,patients_who_have_waited_4_12_hs_from_dta_to_admission,patients_who_have_waited_12_hrs_from_dta_to_admission,emergency_admissions_via_a_e_type_1,emergency_admissions_via_a_e_type_2,emergency_admissions_via_a_e_other_a_e_department,other_emergency_admissions


In [6]:
# Missing values after conversion
clean_df.isnull().sum().sort_values(ascending=False)

period                                                        0
org_code                                                      0
parent_org                                                    0
org_name                                                      0
a_e_attendances_type_1                                        0
a_e_attendances_type_2                                        0
a_e_attendances_other_a_e_department                          0
a_e_attendances_booked_appointments_type_1                    0
a_e_attendances_booked_appointments_type_2                    0
a_e_attendances_booked_appointments_other_department          0
attendances_over_4hrs_type_1                                  0
attendances_over_4hrs_type_2                                  0
attendances_over_4hrs_other_department                        0
attendances_over_4hrs_booked_appointments_type_1              0
attendances_over_4hrs_booked_appointments_type_2              0
attendances_over_4hrs_booked_appointment

In [7]:
negative_values = (clean_df[numeric_cols] < 0).sum()

negative_values[negative_values > 0]

Series([], dtype: int64)

In [14]:
check = clean_df[
["org_name","a_e_attendances_type_1","attendances_over_4hrs_type_1"]
].copy()

check["invalid_waiting_count"] = (check["attendances_over_4hrs_type_1"]> check["a_e_attendances_type_1"])

check[check["invalid_waiting_count"]]

,org_name,a_e_attendances_type_1,attendances_over_4hrs_type_1,invalid_waiting_count


In [16]:
clean_df.to_csv(r"C:\Users\HP\nhs_a&e_performance analysis/Data/Processed/April_2025_cleaned.csv",index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
